# NB02D — Local illumination correction only: paired M2 spatial CV

**One-variable experiment against frozen NB02B M2 (original RGB).** Same four sites/five mosaics (MOS 3–4 excluded), 30 Nadine exhaustive circles, three original spatial folds, 15 m holdout buffer, native 256×256 RGB crops, ImageNet-pretrained ResNet18 U-Net, masked BCE, AdamW 2e−4, 25 epochs, batch 8, same sampling and same held-out reference polygons.

**Only changed input:** apply the **same local, spatially smooth luminance normalization** in *both* chip preparation and full-circle inference. Normalization uses an image-only Gaussian luminance field with a raster halo; it does not require labels, fit site-specific targets, or normalize each crop separately. Fixed *a priori*: sigma=64 native pixels, target luminance=0.45 (RGB in [0,1]), correction exponent=0.5 and gain clipped to [0.85, 1.18]. These are **exploratory fixed settings**, not tuned to the 30 circles. Smooth correction cannot recover information in fully black/shadowed pixels and might reduce species contrast.

**No M1 retraining, no architecture/hyperparameter search, no illumination parameter sweep, no NB02B overwrite.** Training and all results saved to `Outputs/V3/02D_local_illumination_M2/`.

The held-out circles have already been examined in NB02B, so this is an exploratory paired ablation rather than a truly untouched independent confirmation. Run NB02B first on the same annotation version; this notebook checks that version before training.

**Performance note:** Local SSD staging is enabled. The notebook reads each orthomosaic from Google Drive once per staged batch, rather than performing thousands of halo reads over the Drive mount. Existing completed chip-cache batches can be resumed. It suppresses repeated nodata/alpha warnings without changing raster masks.

**Update for future normal runs:** Evaluation now uses NB02B's direct-window center mask, while corrected RGB continues to use the halo. Evaluation caches use a new versioned key; cached model checkpoints and training chips remain reusable. The original recovery-only Block 08 is retained for older in-progress sessions and is not needed for normal fresh runs.


## 01 — Inputs and frozen reference / original-RGB baseline


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn segmentation-models-pytorch
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
BASELINE=ROOT/'Outputs/V3/02B_spatial3fold_no_MOS34'
OUT=ROOT/'Outputs/V3/02D_local_illumination_M2';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
mosaics={m['file']:m for m in A['mosaics'] if m['file']!=EXCLUDED_MOSAIC}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==30
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
assert links.circle_id.is_unique
assert len(mosaics)==5,[(k,v['site']) for k,v in mosaics.items()]
TEST_SITE='SPATIAL_CV'  # training includes all four sites
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('ILLUMINATION ONLY: 3-fold within-site CV, M2 LOCAL vs frozen M2 RAW')
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




print('Mosaics:',[(name,m['site']) for name,m in mosaics.items()])


## 02 — Identical references and pretrained ResNet18 U-Net


In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)>0, (
    f'Unexpected reference geometries: circles={len(circles)} (expected 34), '
    f'Musanga polygons={len(positive)}; do not train until reconciled')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class ResNet18UNet(nn.Module):
    """Original main-branch U-Net: ImageNet-pretrained ResNet18 encoder, RGB in, 1 logit out."""
    def __init__(self):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
                          in_channels=3,classes=1)
    def forward(self,x):
        return self.net(x)[:,0]

class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('ResNet18 U-Net ready (native 256×256 RGB crops).')




## 03 — Spatial exclusion: unchanged three folds and 15 m holdout buffer


In [ ]:
assert links.groupby(['site','fold']).circle_id.nunique().ge(1).all()
assert sorted(links.fold.unique().astype(int).tolist())==[0,1,2]
HOLDOUT=set();DEV=set();BLOCKED={}
split_df=links.iloc[:0].copy()
print('NB01 circle folds:',links.groupby(['site','fold']).size().unstack(fill_value=0).to_dict())


## 04 — Direct raster windows, fixed local illumination correction, resumable M2 chip caches


In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
candidate=candidate[candidate.mosaic.isin(mosaics)].copy()
all_circle_buffers={}
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs'])
    d['tree_positive']=d['trees_local'].loc[d['trees_local'].class01==1]
    d['tree_negative']=d['trees_local'].loc[d['trees_local'].class01==0]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(geometries,name,win):
    """Rasterize geometries intersecting a crop, using the spatial index.

    Accept GeoSeries OR GeoDataFrame; always iterate actual Shapely geometries,
    never GeoDataFrame column names.
    """
    d=geo[name]
    series=geometries.geometry if isinstance(geometries,gpd.GeoDataFrame) else geometries
    assert isinstance(series,gpd.GeoSeries),type(series)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    shape=(int(win.height),int(win.width))
    if len(ids)==0:return np.zeros(shape,dtype=bool)
    selected=series.iloc[ids]
    return rasterize(((g,1) for g in selected.values
                      if g is not None and not g.is_empty),
        out_shape=shape,transform=rasterio.windows.transform(win,d['transform']),
        fill=0,dtype='uint8').astype(bool)

_ACTIVE_RASTER=None
_ACTIVE_NAME=None

# Fixed BEFORE comparing held-out circles; apply identically to training AND inference.
# Keep correction consistent at tile boundaries by reading a 3-sigma halo
# from the full native orthomosaic, rather than normalizing each chip.
import cv2
ILLUM_SIGMA_PX=64.0
ILLUM_HALO_PX=int(np.ceil(3*ILLUM_SIGMA_PX))
ILLUM_REFERENCE=0.45
ILLUM_EXPONENT=0.5
ILLUM_GAIN_MIN=0.85
ILLUM_GAIN_MAX=1.18
ILLUM_VERSION='local_luminance_gaussian_sigma64_target045_power05_gain085_118_v1'

def rgb_for_window(name,win,correction=True):
    """Return same central RGB/valid-pixel crop as NB02B; correction uses a halo.
    The correction is image-only and deterministic in global raster coordinates.
    """
    H=int(win.height);W=int(win.width)
    halo=ILLUM_HALO_PX if correction else 0
    expanded=Window(int(win.col_off)-halo,int(win.row_off)-halo,W+2*halo,H+2*halo)
    def _read(ds):
        return ds.read([1,2,3],window=expanded,boundless=True,masked=True,fill_value=0)
    if _ACTIVE_RASTER is not None and _ACTIVE_NAME==name:
        raw=_read(_ACTIVE_RASTER)
    else:
        with rasterio.open(geo[name]['path']) as ds:raw=_read(ds)
    a=raw.filled(0).astype('float32')/255.0
    masks=np.ma.getmaskarray(raw)
    valid_all=(~np.any(masks,axis=0))
    byte=(a*255.)
    valid_all &= np.any(byte!=0,axis=0)&np.any(byte!=255,axis=0)
    valid_all &= ~((byte[0]==0)&(byte[1]==0)&(byte[2]==255))
    if correction:
        # Weighted smoothing keeps nodata margins from appearing as dark canopy.
        lum=(0.2126*a[0]+0.7152*a[1]+0.0722*a[2]).astype('float32')
        mask=valid_all.astype('float32')
        smooth_num=cv2.GaussianBlur(lum*mask,(0,0),ILLUM_SIGMA_PX,
                                     borderType=cv2.BORDER_REFLECT_101)
        smooth_den=cv2.GaussianBlur(mask,(0,0),ILLUM_SIGMA_PX,
                                     borderType=cv2.BORDER_REFLECT_101)
        local=smooth_num/np.maximum(smooth_den,1e-4)
        gain=np.power(ILLUM_REFERENCE/np.maximum(local,0.05),ILLUM_EXPONENT)
        gain=np.clip(gain,ILLUM_GAIN_MIN,ILLUM_GAIN_MAX)
        a=np.clip(a*gain[None,:,:],0,1)
    img=np.moveaxis(a[:,halo:halo+H,halo:halo+W],0,-1).astype('float32')
    valid=valid_all[halo:halo+H,halo:halo+W]
    return img,valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    pos_tree=burn_window(d['tree_positive'],name,win)
    neg_tree=burn_window(d['tree_negative'],name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def _pack_chips(path,items,signature):
    """Atomic, compressed uint8 cache of already-prepared chips."""
    if not items:return
    tmp=path.with_name(path.name+'.tmp.npz')
    try:
        np.savez_compressed(tmp,key=np.array(signature),
            rgb=np.stack([np.rint(np.clip(r['image'],0,1)*255).astype('uint8') for r in items]),
            target=np.stack([r['target'].astype('uint8') for r in items]),
            weight=np.stack([r['weight'].astype('uint8') for r in items]),
            site=np.array([r['site'] for r in items],dtype='U12'),
            kind=np.array([r['kind'] for r in items],dtype='U16'))
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def _unpack_chips(path,signature):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature
        rgb=z['rgb'];target=z['target'];weight=z['weight']
        sites=z['site'].astype(str);kinds=z['kind'].astype(str)
        assert rgb.ndim==4 and rgb.shape[1:]==(CROP,CROP,3)
        assert target.shape==weight.shape==rgb.shape[:3]
        return [dict(image=rgb[i].astype('float32')/255.,
                     target=target[i].astype('float32'),
                     weight=weight[i].astype('float32'),
                     site=str(sites[i]),kind=str(kinds[i]))
                for i in range(len(rgb))]

import shutil
import warnings
from rasterio.errors import NodataShadowWarning
# Rasterio warning can repeat thousands of times with mosaics containing nodata + alpha.
# No masking behavior is changed by limiting the warning display.
warnings.filterwarnings('once',category=NodataShadowWarning)
from contextlib import contextmanager
LOCAL_STAGE=Path('/content/musanga_02f_stage')
LOCAL_STAGE.mkdir(parents=True,exist_ok=True)
MIN_FREE_AFTER_STAGE=2*1024**3
USE_LOCAL_STAGE=True

@contextmanager
def staged_mosaic(name):
    """Stage one raster on SSD if it fits; use one open raster handle per mosaic."""
    global _ACTIVE_RASTER,_ACTIVE_NAME
    source=Path(geo[name]['path'])
    assert source.exists(),source
    free=shutil.disk_usage('/content').free
    needed=source.stat().st_size
    local=LOCAL_STAGE/(name.replace('/','_'))
    can_stage=USE_LOCAL_STAGE and free>=needed+MIN_FREE_AFTER_STAGE
    selected_path=source
    if can_stage:
        print(f'STAGE {name}: {needed/1e9:.2f} GB from Drive to local SSD '
              f'(available {free/1e9:.1f} GB)',flush=True)
        t0=time.perf_counter()
        tmp=local.with_name(local.name+'.part')
        try:
            with source.open('rb') as inp,tmp.open('wb') as out:
                shutil.copyfileobj(inp,out,length=16*1024*1024)
            assert tmp.stat().st_size==needed
            tmp.replace(local)
            selected_path=local
            print(f'STAGED {name} in {time.perf_counter()-t0:.1f}s',flush=True)
        except Exception as exc:
            tmp.unlink(missing_ok=True)
            local.unlink(missing_ok=True)
            print(f'Local copy failed; using original Drive image: {exc}',flush=True)
    else:
        print(f'STAGE SKIPPED {name}: image {needed/1e9:.2f} GB, '
              f'free SSD {free/1e9:.2f} GB; using one persistent Drive handle',
              flush=True)
    assert _ACTIVE_RASTER is None,'Another mosaic is already open.'
    try:
        with rasterio.open(selected_path) as ds:
            assert ds.count>=3 and ds.transform==geo[name]['transform']
            assert ds.crs==geo[name]['crs']
            _ACTIVE_RASTER=ds
            _ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None
        _ACTIVE_NAME=None
        if selected_path==local:
            local.unlink(missing_ok=True)
            print(f'RELEASED local SSD for {name}',flush=True)

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))



# Training subsets are drawn only from the NB01 fold-specific manifest.
import hashlib

# CONTENT signatures, not just paths/mtime: modified shapefile geometry or DBF
# triggers a new M2 training key and *both* M1/M2 evaluation keys.
def shapefile_content_signature(source):
    p=Path(source)
    if not p.is_file():raise FileNotFoundError(p)
    files=sorted(p.parent.glob(p.stem+'.*')) if p.suffix.lower()=='.shp' else [p]
    files=([x for x in files if x.suffix.lower() in ('.shp','.shx','.dbf','.prj','.cpg')]
           if p.suffix.lower()=='.shp' else [p])
    h=hashlib.sha256()
    for file in files:
        h.update(file.suffix.lower().encode())
        with file.open('rb') as stream:
            for chunk in iter(lambda:stream.read(4*1024*1024),b''):
                h.update(chunk)
    return h.hexdigest()

EXHAUSTIVE_SOURCE=Path(A['exhaustive_annotations']['positive_polygons'])
EXHAUSTIVE_SHA=shapefile_content_signature(EXHAUSTIVE_SOURCE)
print('Exhaustive annotation SHA256:',EXHAUSTIVE_SHA[:20],
      '| polygons loaded:',len(positive),flush=True)

CACHE_DIR=OUT/'chip_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
def cv_fingerprint(fold,model_type):
    sources=[REF/'01_cv_window_manifest.csv',REF/'01_circle_spatial_folds.csv',
             REF/'01_candidate_windows.csv',ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json']
    for name in sorted(mosaics):
        sources.append(Path(mosaics[name]['path']))
    deps=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in sources]
    payload=dict(version='spatial3fold_illumination_v1',illumination=ILLUM_VERSION,fold=int(fold),model=model_type,
                 holdout=sorted(HOLDOUT),dev=sorted(DEV),crop=CROP,seed=SEED,
                 max_partial=MAX_PARTIAL_PER_MOSAIC,
                 max_exhaustive=MAX_EXHAUSTIVE_PER_SITE,buffer_m=15,files=deps,
                 exhaustive_annotations_sha256=EXHAUSTIVE_SHA if model_type=='M2' else None)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def make_cv_cache(model_type,fold):
    key=cv_fingerprint(fold,model_type)
    cache_file=CACHE_DIR/f'{model_type}_fold{fold}_{key[:20]}.npz'
    if cache_file.exists():
        try:
            old=_unpack_chips(cache_file,key)
            assert len(old)>15
            assert all(x['kind']=='partial' for x in old) if model_type=='M1' else True
            print('CACHE HIT',model_type,'fold',fold,len(old),'chips',flush=True)
            return old
        except Exception as e:
            print('Invalid full cache, rebuilding:',repr(e),flush=True)
            cache_file.unlink(missing_ok=True)
    # Same NB01 train candidate windows for both models, with exhaustive windows
    # added exclusively in M2; validation-fold windows are never eligible.
    eligible=splits[(splits.cv_fold==fold)&(splits.split=='train')].copy()
    eligible=eligible[eligible.kind.isin(['partial'] if model_type=='M1' else ['partial','exhaustive'])]
    if model_type=='M2':
        eligible=eligible[(eligible.kind=='partial')|eligible.circle_id.isin(DEV)]
    chosen=[]
    for (name,kind),group in eligible.groupby(['mosaic','kind']):
        cap=MAX_PARTIAL_PER_MOSAIC*(3 if geo[name]['site']=='MOS' else 1) if kind=='partial' else MAX_EXHAUSTIVE_PER_SITE
        group=group.sample(n=min(cap,len(group)),random_state=SEED)
        chosen.append(group)
    assert chosen,'No CV training windows'
    selected=pd.concat(chosen,ignore_index=True)
    assert not selected[selected.kind=='exhaustive'].circle_id.isin(HOLDOUT).any()
    print(model_type,'fold',fold,'selected windows:',selected.groupby(['site','kind']).size().to_dict(),flush=True)
    parts_dir=CACHE_DIR/f'{model_type}_fold{fold}_{key[:20]}'
    parts_dir.mkdir(parents=True,exist_ok=True)
    all_chips=[]
    for name,grp in selected.groupby('mosaic',sort=True):
        records=list(grp.itertuples(index=False))
        with staged_mosaic(name):
            for off in range(0,len(records),100):
                part=parts_dir/f'{name}_{off:04d}.npz'
                if part.exists():
                    try:
                        chunks=_unpack_chips(part,key)
                        all_chips.extend(chunks)
                        continue
                    except Exception:
                        part.unlink(missing_ok=True)
                items=[x for row in records[off:off+100] if (x:=training_crop(row,model_type)) is not None]
                if items:_pack_chips(part,items,key)
                all_chips.extend(items)
                print(model_type,'fold',fold,name,min(off+100,len(records)),'/',len(records),'usable',len(items),flush=True)
    assert len(all_chips)>15,(model_type,fold,len(all_chips))
    assert all(chip['kind']=='partial' for chip in all_chips) if model_type=='M1' else True
    _pack_chips(cache_file,all_chips,key)
    print('CACHE SAVED',model_type,'fold',fold,len(all_chips),'chips',flush=True)
    return all_chips


### Image-only correction preview (before model fitting)

Inspect the *same* raw and corrected window at each site. Correction is computed from the full raster with a 192-pixel halo. This preview does not use exhaustive labels or influence parameter selection. If colours or shadows look distorted, **stop before training** and discuss the fixed correction settings.

In [ ]:
# Preview one valid candidate window per forest, raw vs corrected.
fig,axes=plt.subplots(4,2,figsize=(9,16),layout='constrained')
for row,site in enumerate(['Baego','MOS','Yangambi','Yoko']):
    name=next(k for k,v in mosaics.items() if v['site']==site)
    cand=candidate[(candidate.mosaic==name)&(candidate.kind=='partial')].iloc[0]
    rr=int(cand.row)+(int(cand.height)-CROP)//2
    cc=int(cand.col)+(int(cand.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    with staged_mosaic(name):
        original,_=rgb_for_window(name,win,correction=False)
        corrected,_=rgb_for_window(name,win,correction=True)
    for ax,img,label in [(axes[row,0],original,'Native RGB'),
                          (axes[row,1],corrected,'Locally corrected RGB')]:
        ax.imshow(img);ax.set_title(f'{site} — {label}')
        ax.axis('off')
plt.show();plt.close(fig)


## 05 — M2 only: identical ResNet18 training recipe and three spatial folds


In [ ]:
import time
from contextlib import nullcontext
USE_AMP=(device=='cuda')
# float16 is fast on both T4 and L4; BCE is evaluated in fp32 for stability.
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
print('CUDA mixed precision:',USE_AMP,'| GPU:',torch.cuda.get_device_name(0),flush=True)


def fit_cv(model_type,fold):
    fold_out=BASE_OUT/f'fold_{fold}'
    fold_out.mkdir(parents=True,exist_ok=True)
    key=cv_fingerprint(fold,model_type)
    # Versioned M2 models cannot silently reuse old Yangambi labels.
    # M1 uses partial labels only. Reuse the original completed M1 checkpoint
    # under the explicit assumption that only exhaustive POSITIVE polygons changed.
    legacy_m1=fold_out/f'fold_{fold}'/'M1_resnet18_spatialCV.pt'
    ckpt=(legacy_m1 if model_type=='M1' and legacy_m1.exists() else
          fold_out/f'M1_resnet18_spatialCV.pt' if model_type=='M1' else
          fold_out/f'M2_local_illumination_{key[:20]}.pt')
    torch.manual_seed(SEED)
    model=ResNet18UNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Reusing spatial-CV checkpoint:',ckpt,flush=True)
        if model_type=='M1':
            print('M1 reuse assumes partial tree annotations, circle boundaries, '
                  'fold allocation and ROI unchanged.',flush=True)
        return model.eval()
    chips=make_cv_cache(model_type,fold)
    loader=DataLoader(CropDataset(chips),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    optim=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    local_scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
    history=[]
    for ep in range(EPOCHS):
        model.train();losses=[];started=time.perf_counter()
        for x,y,w in loader:
            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True);w=w.to(device,non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
                logits=model(x)
            loss=masked_loss(logits.float(),y,w)
            local_scaler.scale(loss).backward()
            local_scaler.step(optim);local_scaler.update()
            losses.append(float(loss.detach()))
        avg=float(np.mean(losses))
        history.append(dict(epoch=ep+1,masked_BCE=avg,seconds=time.perf_counter()-started))
        pd.DataFrame(history).to_csv(fold_out/f'{model_type}_loss.csv',index=False)
        print(f'{model_type} fold {fold} epoch {ep+1:02d}/{EPOCHS} BCE={avg:.4f} {history[-1]["seconds"]:.1f}s',flush=True)
    torch.save(model.state_dict(),ckpt)
    del chips
    print('Saved:',ckpt,flush=True)
    return model.eval()


## 06 — Held-out inference uses identical illumination transform and circles


In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
VAL_WINDOWS=VAL_WINDOWS[VAL_WINDOWS.circle_id.isin(links.circle_id)].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_spatialCV_circle_raw_{EVAL_KEY[:20]}.csv'
    histfile=OUT/f'{model_name}_spatialCV_histograms_{EVAL_KEY[:20]}.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=tn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            # Illumination-corrected RGB is produced with spatial halo. For
            # evaluation, adopt EXACTLY the baseline NB02B center-pixel mask.
            # MOS nodata + alpha combinations may mask differently during
            # boundless halo reads; no test pixels may be added/dropped.
            rgb,_halo_valid=rgb_for_window(name,win)
            with rasterio.open(geo[name]['path']) as ds:
                center_raw=ds.read([1,2,3],window=win,masked=True)
            original=center_raw.filled(0).astype('float32')
            valid=~np.any(np.ma.getmaskarray(center_raw),axis=0)
            valid &= np.any(original!=0,axis=0)&np.any(original!=255,axis=0)
            valid &= ~((original[0]==0)&(original[1]==0)&(original[2]==255))
            assert valid.shape==rgb.shape[:2]
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            tn+=int(np.sum((~binary)&(y==0)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         tp=tp,fp=fp,fn=fn,tn=tn,
                         iou_threshold_05=tp/max(1,tp+fp+fn),
                         precision_threshold_05=tp/max(1,tp+fp),
                         recall_threshold_05=tp/max(1,tp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert frame.circle_id.is_unique
    assert set(frame.circle_id.astype(int))==set(split_df.circle_id.astype(int))
    assert len(frame)==len(split_df)
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms



In [ ]:
# PRIMARY spatial-CV experiment: site-stratified circle folds (not LOSO).
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import gc
SITES=['Baego','MOS','Yangambi','Yoko']
FOLDS=[0,1,2]
BASE_OUT=OUT
# Ensure baseline uses EXACTLY the annotations being evaluated here.
baseline_version_file=BASELINE/'02B_latest_reference_version.json'
assert baseline_version_file.exists(),('Run NB02B first:',baseline_version_file)
baseline_version=json.loads(baseline_version_file.read_text())
assert baseline_version['sha256']==EXHAUSTIVE_SHA,(
    'NB02B annotation version differs: rerun NB02B on current references',
    baseline_version['sha256'],EXHAUSTIVE_SHA)
BASELINE_CSV=BASELINE/'02B_spatialCV_all30_paired.csv'
assert BASELINE_CSV.exists(),BASELINE_CSV
BASELINE_M2=pd.read_csv(BASELINE_CSV)
BASELINE_M2=BASELINE_M2[BASELINE_M2.model=='M2'].copy()
assert len(BASELINE_M2)==30 and BASELINE_M2.circle_id.is_unique

# Different exhaustive polygon bytes => different evaluation cache signature.
# Also account for circle boundaries / ROI / validation windows (if modified).
def evaluation_fingerprint(fold,model_type):
    validation_sources=[Path(A['exhaustive_annotations']['complete_roi_boundaries']),
                        Path(A['roi']),REF/'01_candidate_windows.csv',
                        REF/'01_circle_spatial_folds.csv']
    h=hashlib.sha256()
    h.update(json.dumps(dict(fold=int(fold),model=model_type,
              source_sha=EXHAUSTIVE_SHA,
              training_key=cv_fingerprint(fold,model_type)),sort_keys=True).encode())
    for p in validation_sources:
        h.update(shapefile_content_signature(p).encode())
    # A new signature forbids reuse of old halo-mask evaluation results.
    h.update(b'nb02d_baseline_center_mask_v3')
    return h.hexdigest()

print('Reference annotation version:',EXHAUSTIVE_SHA[:20],flush=True)
assert sorted(links.fold.unique().astype(int))==FOLDS
assert links.groupby(['site','fold']).circle_id.nunique().ge(1).all()

ALL_SCORES=[]
ALL_HISTS={}
for cv_fold in FOLDS:
    FOLD=int(cv_fold)
    HOLDOUT=set(links.loc[links.fold==FOLD,'circle_id'].astype(int))
    DEV=set(links.loc[links.fold!=FOLD,'circle_id'].astype(int))
    assert HOLDOUT.isdisjoint(DEV) and len(HOLDOUT|DEV)==30
    split_df=links[links.fold==FOLD][['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']='held_out_spatial_fold'
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        # Include all validation circles geometrically, not just own-mosaic circles.
        BLOCKED[name]=[g.buffer(15) for g in cs[cs.original_row.isin(HOLDOUT)].geometry]
    OUT=BASE_OUT/f'fold_{FOLD}'
    OUT.mkdir(parents=True,exist_ok=True)
    print('\n===== SPATIAL CV FOLD',FOLD,'holdout circles',len(HOLDOUT),'training circles',len(DEV),'=====',flush=True)
    for model_type in ['M2']:
        EVAL_KEY=evaluation_fingerprint(FOLD,model_type)
        print('Evaluation signature',model_type,FOLD,EVAL_KEY[:20],flush=True)
        model=fit_cv(model_type,FOLD)
        result,h=run_all_circles(model,model_type)
        assert set(result.circle_id.astype(int))==HOLDOUT
        result['fold']=FOLD
        ALL_SCORES.append(result)
        for cid in HOLDOUT:
            for suffix in ['count','positive']:
                ALL_HISTS[f'{model_type}_{cid}_{suffix}']=h[f'{cid}_{suffix}']
        pd.concat(ALL_SCORES,ignore_index=True).to_csv(BASE_OUT/'02B_spatialCV_progress.csv',index=False)
        print('Completed',model_type,'fold',FOLD,'circles',len(result),flush=True)
        del model;torch.cuda.empty_cache();gc.collect()

SCORES=pd.concat(ALL_SCORES,ignore_index=True)
assert len(SCORES)==30 and SCORES.circle_id.is_unique
SCORES.to_csv(BASE_OUT/'02D_M2_LOCAL_spatialCV_all30.csv',index=False)
np.savez_compressed(BASE_OUT/'02D_M2_LOCAL_histograms.npz',**ALL_HISTS)

raw=BASELINE_M2.set_index('circle_id').sort_index()
loc=SCORES.set_index('circle_id').sort_index()
assert raw.index.equals(loc.index)
assert (raw.site==loc.site).all() and (raw.fold==loc.fold).all()
assert np.array_equal(raw.valid_pixels.to_numpy(),loc.valid_pixels.to_numpy()),( 
    'Baseline and illumination valid-pixel counts differ; do not compare')
assert np.allclose(raw.observed_fraction,loc.observed_fraction,atol=1e-5),(
    'Baseline and illumination reference values differ; do not compare')

comparison=pd.DataFrame({'circle_id':raw.index,'site':raw.site.values,
    'fold':raw.fold.values,'observed_fraction':raw.observed_fraction.values,
    'raw_fraction':raw.predicted_fraction.values,
    'local_fraction':loc.predicted_fraction.values,
    'raw_pixel_f1':raw.f1_threshold_05.values,
    'local_pixel_f1':loc.f1_threshold_05.values,
    'raw_pixel_iou':raw.iou_threshold_05.values,
    'local_pixel_iou':loc.iou_threshold_05.values})
comparison['raw_abs_error_pp']=abs(comparison.raw_fraction-comparison.observed_fraction)
comparison['local_abs_error_pp']=abs(comparison.local_fraction-comparison.observed_fraction)
comparison['improvement_pp']=comparison.raw_abs_error_pp-comparison.local_abs_error_pp
comparison.to_csv(BASE_OUT/'02D_paired_raw_vs_local_30circles.csv',index=False)

def metric_line(sub,method,site):
    obs=sub.observed_fraction.to_numpy()
    pred=sub.predicted_fraction.to_numpy()
    tp,fp,fn=[float(sub[k].sum()) for k in ('tp','fp','fn')]
    return dict(method=method,site=site,n=len(sub),
        R2=r2_score(obs,pred) if len(sub)>1 else np.nan,
        MAE_pp=mean_absolute_error(obs,pred),
        RMSE_pp=np.sqrt(mean_squared_error(obs,pred)),
        bias_pp=float(np.mean(pred-obs)),
        pixel_F1=2*tp/max(1,2*tp+fp+fn),
        pixel_IoU=tp/max(1,tp+fp+fn),
        pixel_precision=tp/max(1,tp+fp),
        pixel_recall=tp/max(1,tp+fn))
rows=[]
for method,frame in [('RAW_NB02B',BASELINE_M2),('LOCAL_NB02D',SCORES)]:
    for site in ['ALL']+SITES:
        subset=frame if site=='ALL' else frame[frame.site==site]
        rows.append(metric_line(subset,method,site))
PERFORMANCE=pd.DataFrame(rows)
PERFORMANCE.to_csv(BASE_OUT/'02D_paired_performance_by_site.csv',index=False)
display(PERFORMANCE.round(4))
print('LOCAL M2 better in',int((comparison.improvement_pp>0).sum()),
      'of 30 circles, same reference and same held-out folds.')
(BASE_OUT/'02D_illumination_config.json').write_text(json.dumps(dict(
    version=ILLUM_VERSION,baseline_annotation_sha256=EXHAUSTIVE_SHA,
    gaussian_sigma_px=ILLUM_SIGMA_PX,halo_px=ILLUM_HALO_PX,
    target_luminance=ILLUM_REFERENCE,exponent=ILLUM_EXPONENT,
    gain_min=ILLUM_GAIN_MIN,gain_max=ILLUM_GAIN_MAX,
    folds=FOLDS,only_model='M2'),indent=2))
print('ILLUMINATION ABLATION COMPLETE:',BASE_OUT)


## 07 — Paired visual results: raw versus local illumination

No model tuning or threshold selection on the held-out circles. Circle labels denote the independent holdout reference. The scatter plot shows fractional-cover agreement, the paired plot shows change in absolute error, and site metrics show whether the correction is effective in shaded or old-growth conditions. Any brightness-stratified pixel comparison is a **separate diagnostic** requiring a saved per-pixel probability map, not inferred from aggregate CSVs.

In [ ]:
# Read saved paired results; valid even in a fresh session after setup.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display
FIG_DIR=BASE_OUT/'figures';FIG_DIR.mkdir(parents=True,exist_ok=True)
colors={'Baego':'#3168B4','MOS':'#E28E23','Yangambi':'#27856A','Yoko':'#AB477B'}
fig,axes=plt.subplots(1,2,figsize=(13,6),sharex=True,sharey=True,layout='constrained')
for ax,(col,label) in zip(axes,[('raw_fraction','Original RGB · NB02B M2'),
                                 ('local_fraction','Local illumination · NB02D M2')]):
    for site in SITES:
        d=comparison[comparison.site==site]
        ax.scatter(d.observed_fraction,d[col],color=colors[site],s=62,alpha=.85,edgecolor='white')
        for q in d.itertuples():
            ax.annotate(str(int(q.circle_id)),(q.observed_fraction,getattr(q,col)),
                        xytext=(4,3),textcoords='offset points',fontsize=8,color=colors[site])
    ax.plot([0,100],[0,100],'--',color='0.35',lw=1)
    ax.set(xlim=(-2,103),ylim=(-2,103),aspect='equal',
           xlabel='Observed Musanga cover (%)',ylabel='Predicted Musanga cover (%)',
           title=label)
    ax.grid(alpha=.16)
fig.legend(handles=[Line2D([0],[0],marker='o',color='none',
           markerfacecolor=colors[site],markersize=8,label=site)
           for site in SITES],loc='outside lower center',ncol=4,frameon=False)
fig.suptitle('Held-out fractional cover · same 30 circles')
for ext in ('png','pdf'):fig.savefig(FIG_DIR/f'fig01_paired_cover.{ext}',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(7,6.5),layout='constrained')
maximum=1+max(comparison.raw_abs_error_pp.max(),comparison.local_abs_error_pp.max())
for site in SITES:
    d=comparison[comparison.site==site]
    ax.scatter(d.raw_abs_error_pp,d.local_abs_error_pp,s=65,color=colors[site],label=site)
    for q in d.itertuples():
        ax.annotate(str(int(q.circle_id)),(q.raw_abs_error_pp,q.local_abs_error_pp),
                    xytext=(4,3),textcoords='offset points',fontsize=8)
ax.plot([0,maximum],[0,maximum],'--',color='0.35')
ax.set(xlim=(0,maximum),ylim=(0,maximum),aspect='equal',
       xlabel='Original RGB absolute error (pp)',
       ylabel='Locally corrected RGB absolute error (pp)',
       title='Below 1:1 = local illumination improves cover')
ax.grid(alpha=.16);ax.legend(frameon=False)
for ext in ('png','pdf'):fig.savefig(FIG_DIR/f'fig02_paired_errors.{ext}',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)

print('Site comparison, raw versus local illumination:')
display(PERFORMANCE.round(4))
print('Figures:',FIG_DIR)


## 08 — Recovery after evaluation-mask mismatch (NO TRAINING)

**Use this cell if the original run completed all 3 folds but failed** with `Baseline and illumination reference values differ`. In the **same runtime** that produced the failure, execute only the cell below. It reuses the existing M2 checkpoints, compares identical RGB pixel masks to NB02B, and reruns **inference only** on the 30 validation circles. The previously trained RGB-illumination models and existing chip caches are not modified.

**Cause being addressed:** Rasterio `boundless=True` halo reads can give a different center mask from direct non-boundless reads of MOS rasters with NoData/alpha metadata. The corrected imagery retains its original local Gaussian correction, but the evaluation population uses the original NB02B center mask. No thresholds or parameters are tuned to improve scores.

If `observed_fraction` or `valid_pixels` still disagree, the cell prints differences and stops rather than misreporting comparative metrics. Old results are kept under their previous signatures; repaired recovery results use a distinct `maskmatched_v2` evaluation signature. Because original training chip weights were obtained via halo reads, a successful evaluation-mask match makes **evaluation** comparable but does not retroactively equate the training sample validity masks.


In [ ]:
# RUN THIS CELL ONLY after the previous complete main fold loop finished.
# It reuses trained checkpoints; it MUST NOT train models or rebuild chips.
import gc, hashlib, json, numpy as np, pandas as pd, rasterio, torch
from pathlib import Path
from rasterio.windows import Window

assert 'BASELINE_M2' in globals() and 'EXHAUSTIVE_SHA' in globals(), (
    'First run setup and definition cells (01–06). Do not rerun the original training loop.')
assert baseline_version['sha256']==EXHAUSTIVE_SHA

# Save the previous halo-based illumination function; preserve its RGB correction.
if '_illumination_rgb_halo_original' not in globals():
    _illumination_rgb_halo_original=rgb_for_window

def rgb_for_window(name,win,correction=True):
    """NB02B-equivalent CENTER valid mask, locally corrected halo RGB.
    The same valid mask is used to score reference & predictions.
    """
    img,_halo_valid=_illumination_rgb_halo_original(name,win,correction=correction)
    if _ACTIVE_RASTER is not None and _ACTIVE_NAME==name:
        raw=_ACTIVE_RASTER.read([1,2,3],window=win,masked=True)
    else:
        with rasterio.open(geo[name]['path']) as ds:
            raw=ds.read([1,2,3],window=win,masked=True)
    orig=raw.filled(0).astype('float32')
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    valid &= np.any(orig!=0,axis=0)&np.any(orig!=255,axis=0)
    valid &= ~((orig[0]==0)&(orig[1]==0)&(orig[2]==255))
    assert valid.shape==img.shape[:2],(name,win,valid.shape,img.shape)
    return img,valid

# Leave original cached inference files alone: use new signature.
ALL_SCORES_RECOVERY=[]
FOLDS=[0,1,2]
SITES=['Baego','MOS','Yangambi','Yoko']
BASE_OUT=ROOT/'Outputs/V3/02D_local_illumination_M2'
for cv_fold in FOLDS:
    FOLD=int(cv_fold)
    HOLDOUT=set(links.loc[links.fold==FOLD,'circle_id'].astype(int))
    DEV=set(links.loc[links.fold!=FOLD,'circle_id'].astype(int))
    split_df=links[links.fold==FOLD][['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']='held_out_spatial_fold'
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        BLOCKED[name]=[g.buffer(15) for g in cs[cs.original_row.isin(HOLDOUT)].geometry]
    OUT=BASE_OUT/f'fold_{FOLD}'
    model_type='M2'
    key=cv_fingerprint(FOLD,model_type)
    checkpoint=OUT/f'M2_local_illumination_{key[:20]}.pt'
    assert checkpoint.exists(),('TRAINING ALREADY COMPLETE REQUIRED. No retraining:',checkpoint)
    EVAL_KEY=hashlib.sha256((
        evaluation_fingerprint(FOLD,model_type)+'_center_maskmatched_v2').encode()
    ).hexdigest()
    print('Rescoring fold',FOLD,'only; checkpoint:',checkpoint.name,flush=True)
    model=ResNet18UNet().to(device)
    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True))
    model.eval()
    score,_=run_all_circles(model,model_type)
    score['fold']=FOLD
    assert set(score.circle_id.astype(int))==HOLDOUT
    ALL_SCORES_RECOVERY.append(score)
    del model;gc.collect()
    if device=='cuda':torch.cuda.empty_cache()

SCORES_RECOVERY=pd.concat(ALL_SCORES_RECOVERY,ignore_index=True)
raw=BASELINE_M2.set_index('circle_id').sort_index()
loc=SCORES_RECOVERY.set_index('circle_id').sort_index()
assert raw.index.equals(loc.index)
check=pd.DataFrame({
    'circle_id':raw.index,
    'site':raw.site.values,
    'baseline_n':raw.valid_pixels.values,
    'local_n':loc.valid_pixels.values,
    'baseline_observed':raw.observed_fraction.values,
    'local_observed':loc.observed_fraction.values,
    'observed_difference_pp':loc.observed_fraction.values-raw.observed_fraction.values
})
display(check.loc[(check.baseline_n!=check.local_n)|
    (check.observed_difference_pp.abs()>1e-5)].round(7))
assert np.array_equal(raw.valid_pixels.to_numpy(),loc.valid_pixels.to_numpy()), (
    'Different evaluation pixel populations — do NOT compare')
assert np.allclose(raw.observed_fraction,loc.observed_fraction,atol=1e-5),(
    'Different observed cover despite matched valid pixel counts — do NOT compare')
assert (raw.site==loc.site).all() and (raw.fold==loc.fold).all()

# Safe paired comparison: matching observed fraction, pixel counts, and folds.
comparison=pd.DataFrame({'circle_id':raw.index,'site':raw.site.values,
    'fold':raw.fold.values,'observed_fraction':raw.observed_fraction.values,
    'raw_fraction':raw.predicted_fraction.values,
    'local_fraction':loc.predicted_fraction.values,
    'raw_pixel_f1':raw.f1_threshold_05.values,
    'local_pixel_f1':loc.f1_threshold_05.values,
    'raw_pixel_iou':raw.iou_threshold_05.values,
    'local_pixel_iou':loc.iou_threshold_05.values})
comparison['raw_abs_error_pp']=(comparison.raw_fraction-comparison.observed_fraction).abs()
comparison['local_abs_error_pp']=(comparison.local_fraction-comparison.observed_fraction).abs()
comparison['improvement_pp']=comparison.raw_abs_error_pp-comparison.local_abs_error_pp
comparison.to_csv(BASE_OUT/'02D_paired_raw_vs_local_30circles_MASKMATCHED.csv',index=False)
SCORES_RECOVERY.to_csv(BASE_OUT/'02D_M2_LOCAL_spatialCV_all30_MASKMATCHED.csv',index=False)

from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
def recovery_metrics(frame,method,site):
    y=frame.observed_fraction.to_numpy()
    pred=frame.predicted_fraction.to_numpy()
    tp,fp,fn=[float(frame[c].sum()) for c in ['tp','fp','fn']]
    return dict(method=method,site=site,n=len(frame),
                R2=r2_score(y,pred),MAE_pp=mean_absolute_error(y,pred),
                RMSE_pp=np.sqrt(mean_squared_error(y,pred)),
                bias_pp=float((pred-y).mean()),
                pixel_F1=2*tp/max(1,2*tp+fp+fn),
                pixel_IoU=tp/max(1,tp+fp+fn),
                pixel_precision=tp/max(1,tp+fp),
                pixel_recall=tp/max(1,tp+fn))
rows=[]
for method,frame in [('RAW_NB02B',BASELINE_M2),('LOCAL_NB02D_MASKMATCHED',SCORES_RECOVERY)]:
    for site in ['ALL']+SITES:
        subset=frame if site=='ALL' else frame[frame.site==site]
        rows.append(recovery_metrics(subset,method,site))
PERFORMANCE=pd.DataFrame(rows)
PERFORMANCE.to_csv(BASE_OUT/'02D_paired_performance_MASKMATCHED.csv',index=False)
display(PERFORMANCE.round(4))
print('LOCAL M2 smaller absolute error in',int((comparison.improvement_pp>0).sum()),
      'of 30 circles. Finished without training.')
